# NODE LIGHTUP ANALYSIS 18NOV2025

# Import Libraries

In [1]:
import h5py
import re
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras.models import load_model
import os

2026-10-28 13:59:41.986644: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-28 13:59:46.402686: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-28 13:59:56.547805: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# Import the data to be used for the Node Light Up Technique

In [2]:
# Import expression data and transcript data from filtered h5 file
filtered_file_tf = h5py.File("/scratch/by2372/pipeline/ARCHS4/filtered_ARCHS4_TFs.h5", 'r')
ensembl_id = filtered_file_tf['ensembl_id'][...]
symbol = filtered_file_tf['symbol'][...]
expression = filtered_file_tf['data/filtered_expression'][...]
filtered_file_tf.close()

expression = np.log2(expression + 1)

# Read all names and decode bytes to strings
if isinstance(ensembl_id[0], bytes):
    ensembl_id = [s.decode('utf-8') for s in ensembl_id]

# Read all names and decode bytes to strings
if isinstance(symbol[0], bytes):
    symbol = [s.decode('utf-8') for s in symbol]

# Make the transcript ID be the symbol _ ensembl ID
transcript_ID = []
for idx, sym in enumerate(symbol):
    ID = sym + "_" + ensembl_id[idx]
    transcript_ID.append(ID)

In [3]:
expression.shape

(13146, 63324)

In [4]:
len(transcript_ID)

13146

# Implement the Node Light Up technique as a function

In [5]:
def node_light_up(model_name,tf_names,express_mat,cell_type):
    # load train keras model
    model = load_model('/scratch/by2372/pipeline/ARCHS4/' + model_name + '_trained_NN.keras')
    # create square matrix of zeros. Number of rows and cols is number of TFs
    inp = np.zeros((model.input_shape[1],model.input_shape[1])).T
    # Import cell class names 
    class_names = np.load('/scratch/by2372/pipeline/ARCHS4/' + model_name + '_class_names.npy')
    
    cell_type_index = np.where(class_names == cell_type)[0][0]
    # Generate baseline TF expression values for the inp square matrix
    tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)
    for i in range(inp.shape[1]):
        inp[i, :] = tf_data.values
    # Generate doubled and halved expression for TFs. Only diagonals of the baseline square matrix is doubled or halved
    half = inp.copy()
    double = inp.copy()
    half[np.eye(half.shape[0]).astype(bool)] = 0.5*half[np.eye(half.shape[0]).astype(bool)]
    double[np.eye(double.shape[0]).astype(bool)] = 2*double[np.eye(double.shape[0]).astype(bool)]

    background_pred = model.predict(inp)
    background_idx = np.argmax(background_pred, axis=1)
    background_labels = class_names[background_idx]
    
    double_pred = model.predict(double)
    double_idx = np.argmax(double_pred, axis=1)
    double_labels = class_names[double_idx]
    
    half_pred = model.predict(half)
    half_idx = np.argmax(half_pred, axis=1)
    half_labels = class_names[half_idx]

    double_background = double_pred[:,cell_type_index] - background_pred[:,cell_type_index]
    half_background = half_pred[:,cell_type_index] - background_pred[:,cell_type_index]

    score = (double_background - half_background) / 2
    score = pd.DataFrame(score, index=tf_names, columns=[cell_type])
    score_sorted = score.sort_values(by=cell_type, ascending=True)
    
    return score_sorted

    
    

# Feature Selection for Unbalanced

In [6]:
unbalanced_features = node_light_up("unbalanced",transcript_ID,expression,"Macrophage")

I0000 00:00:1793210456.499723 2980960 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 43516 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:c2:00.0, compute capability: 8.9
/state/partition1/job-21241543/ipykernel_2980960/198994739.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)
2026-10-28 14:01:06.689873: I external/local_xla/xla/service/service.cc:163] XLA service 0x1508e8017b60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-10-28 14:01:06.689895: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA L40S, Compute Capability 8.9
2026-10-28 14:01:06.797477: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-10-28 14:01:07.053148: I external/loc

173/411 ━━━━━━━━━━━━━━━━━━━━ 0s 882us/step

I0000 00:00:1793210470.843604 3039224 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


395/411 ━━━━━━━━━━━━━━━━━━━━ 0s 898us/step

2026-10-28 14:01:11.251026: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-10-28 14:01:11.459383: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_9', 4 bytes spill stores, 4 bytes spill loads

2026-10-28 14:01:12.694118: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_18', 8 bytes spill stores, 8 bytes spill loads

2026-10-28 14:01:13.285900: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_18', 68 bytes spill s

411/411 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 880us/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 896us/step


In [7]:
unbalanced_features.head(n=20) 

,Macrophage
HMGA2_ENST00000403681,-0.003709
TCF7L1_ENST00000282111,-0.002523
SOX2_ENST00000325404,-0.002374
ARID3A_ENST00000263620,-0.002349
KLF6_ENST00000542957,-0.002255
POU5F1B_ENST00000465342,-0.002212
NFIL3_ENST00000297689,-0.002126
ZNF121_ENST00000320451,-0.001978
SON_ENST00000475072,-0.001894
ATF4_ENST00000674920,-0.001891


In [8]:
for idx, t in enumerate(unbalanced_features.index):
    if "ZNF385A" in str(t):
        print(idx)

85
201
892
1242
2043
2348
2416
5122
5198
6388
9519
9575
11482
11593
12175


In [9]:
for idx, t in enumerate(unbalanced_features.index):
    if "CEBPA" in str(t):
        print(idx)

200


In [10]:
for idx, t in enumerate(unbalanced_features.index):
    if "CEBPB" in str(t):
        print(idx)

13031


# Feature Selection for Random Oversampler

In [11]:
RO_features = node_light_up("RO",transcript_ID,expression,"Macrophage")

/state/partition1/job-21241543/ipykernel_2980960/198994739.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step    
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 853us/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 864us/step


In [12]:
RO_features.head(n=20)

,Macrophage
PBX1_ENST00000420696,-0.000086
MYC_ENST00000652288,-0.000077
JUND_ENST00000252818,-0.000074
SOX2_ENST00000325404,-0.000067
NME2_ENST00000503064,-0.000065
ETS2_ENST00000662305,-0.000062
XBP1_ENST00000216037,-0.000059
ZNF22_ENST00000298299,-0.000058
GTF3A_ENST00000419181,-0.000058
DRAP1_ENST00000531121,-0.000057


In [13]:
for idx, t in enumerate(RO_features.index):
    if "ZNF385A" in str(t):
        print(idx)

2192
2242
3588
5084
5312
6436
6797
7742
8349
8372
8853
11023
11317
11347
11683


In [14]:
for idx, t in enumerate(RO_features.index):
    if "CEBPA" in str(t):
        print(idx)

2692


In [15]:
for idx, t in enumerate(RO_features.index):
    if "CEBPB" in str(t):
        print(idx)

12336


# Feature Selection for SMOTE

In [16]:
SMOTE_features = node_light_up("SMOTE",transcript_ID,expression,"Macrophage")

/state/partition1/job-21241543/ipykernel_2980960/198994739.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step    
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 868us/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 888us/step


In [17]:
SMOTE_features.head(n=20)

,Macrophage
SOX2_ENST00000325404,-0.001149
SOX4_ENST00000244745,-0.001038
SON_ENST00000475072,-0.000998
SOX11_ENST00000322002,-0.000842
HMGA2_ENST00000403681,-0.000755
KLF13_ENST00000621843,-0.000607
REPIN1_ENST00000425389,-0.000604
CAMTA1_ENST00000490738,-0.000591
ARID3A_ENST00000263620,-0.000578
DRAP1_ENST00000312515,-0.000542


In [18]:
for idx, t in enumerate(SMOTE_features.index):
    if "ZNF385A" in str(t):
        print(idx)

301
704
1013
1624
1950
3915
5277
5553
5556
8531
8931
9018
10932
11397
11752


In [19]:
for idx, t in enumerate(SMOTE_features.index):
    if "CEBPA" in str(t):
        print(idx)

489


In [20]:
for idx, t in enumerate(SMOTE_features.index):
    if "CEBPB" in str(t):
        print(idx)

12006


# Feature Selection for BLSMOTE

In [21]:
BLSMOTE_features = node_light_up("BLSMOTE",transcript_ID,expression,"Macrophage")

/state/partition1/job-21241543/ipykernel_2980960/198994739.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step    
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 852us/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 875us/step


In [22]:
BLSMOTE_features.head(n=20)

,Macrophage
SOX2_ENST00000325404,-0.000185
SOX4_ENST00000244745,-0.000135
HMGA1_ENST00000347617,-0.000078
CAMTA1_ENST00000490738,-0.000074
HIF1A_ENST00000557446,-0.000072
ARID3B_ENST00000346246,-0.000069
SOX11_ENST00000322002,-0.000066
HES1_ENST00000476918,-0.000065
EGR1_ENST00000239938,-0.000064
JUND_ENST00000600972,-0.000062


In [23]:
for idx, t in enumerate(BLSMOTE_features.index):
    if "ZNF385A" in str(t):
        print(idx)

172
779
1286
1380
2207
2508
2751
2827
4513
10818
10970
11101
11260
11758
13003


In [24]:
for idx, t in enumerate(BLSMOTE_features.index):
    if "CEBPA" in str(t):
        print(idx)

12720


In [25]:
for idx, t in enumerate(BLSMOTE_features.index):
    if "CEBPB" in str(t):
        print(idx)

13002


## Monocytes Results

In [34]:
unbalanced_features = node_light_up("unbalanced",transcript_ID,expression,"Monocyte")
unbalanced_features.head(n=20) 

/state/partition1/job-21241543/ipykernel_2980960/198994739.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step    
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 869us/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 885us/step


,Monocyte
SOX4_ENST00000244745,-0.000034
HMGA1_ENST00000347617,-0.000029
ZNF706_ENST00000519744,-0.000026
ZNF121_ENST00000320451,-0.000022
HMGA2_ENST00000403681,-0.000022
TSC22D1_ENST00000261489,-0.000021
LIN28A_ENST00000254231,-0.000019
ZNF302_ENST00000507959,-0.000019
SOX2_ENST00000325404,-0.000018
BHLHE41_ENST00000242728,-0.000018


In [35]:
for idx, t in enumerate(unbalanced_features.index):
    if "ZNF385A" in str(t):
        print(idx)

406
1117
1206
1908
2166
2243
3110
3119
3274
3601
5291
7576
9153
10709
11996


In [36]:
for idx, t in enumerate(unbalanced_features.index):
    if "CEBPA" in str(t):
        print(idx)

726


In [37]:
for idx, t in enumerate(unbalanced_features.index):
    if "CEBPB" in str(t):
        print(idx)

63


In [38]:
RO_features = node_light_up("RO",transcript_ID,expression,"Monocyte")
RO_features.head(n=20) 

/state/partition1/job-21241543/ipykernel_2980960/198994739.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step    
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 875us/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 895us/step


,Monocyte
JUN_ENST00000678696,-0.000317
ETV5_ENST00000306376,-0.000311
MYBL2_ENST00000217026,-0.000279
BHLHE41_ENST00000242728,-0.000266
YBX3_ENST00000279550,-0.000254
ZNF420_ENST00000589461,-0.000252
HMGN3_ENST00000275036,-0.000251
CREB3_ENST00000353704,-0.000250
TSC22D1_ENST00000261489,-0.000238
TCF7L2_ENST00000534894,-0.000237


In [39]:
for idx, t in enumerate(RO_features.index):
    if "ZNF385A" in str(t):
        print(idx)

164
636
1201
1314
1982
2330
5423
7162
7169
7281
7556
7930
8838
11479
11687


In [40]:
for idx, t in enumerate(RO_features.index):
    if "CEBPA" in str(t):
        print(idx)

9124


In [41]:
for idx, t in enumerate(RO_features.index):
    if "CEBPB" in str(t):
        print(idx)

188


In [42]:
SMOTE_features = node_light_up("SMOTE",transcript_ID,expression,"Monocyte")
SMOTE_features.head(n=20) 

/state/partition1/job-21241543/ipykernel_2980960/198994739.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step    
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 831us/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 852us/step


,Monocyte
SOX4_ENST00000244745,-0.000089
SOX11_ENST00000322002,-0.000057
HMGA2_ENST00000403681,-0.000052
SOX2_ENST00000325404,-0.000050
CAMTA1_ENST00000490738,-0.000047
SON_ENST00000475072,-0.000044
ARID3A_ENST00000263620,-0.000042
CXXC5_ENST00000302517,-0.000037
NME2_ENST00000512737,-0.000036
KLF13_ENST00000621843,-0.000035


In [43]:
for idx, t in enumerate(SMOTE_features.index):
    if "ZNF385A" in str(t):
        print(idx)

228
634
790
1024
1624
2194
3324
3622
3761
5281
5606
5945
7722
7966
9967


In [44]:
for idx, t in enumerate(SMOTE_features.index):
    if "CEBPA" in str(t):
        print(idx)

82


In [45]:
for idx, t in enumerate(SMOTE_features.index):
    if "CEBPB" in str(t):
        print(idx)

5398


In [46]:
BLSMOTE_features = node_light_up("BLSMOTE",transcript_ID,expression,"Monocyte")
BLSMOTE_features.head(n=20) 

/state/partition1/job-21241543/ipykernel_2980960/198994739.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step    
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 833us/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 847us/step


,Monocyte
SOX2_ENST00000325404,-0.000252
HES1_ENST00000476918,-0.000160
STAT2_ENST00000556539,-0.000138
DNMT1_ENST00000586988,-0.000131
TOPORS_ENST00000379858,-0.000116
HIF1A_ENST00000557446,-0.000111
PBX1_ENST00000420696,-0.000096
FOSL2_ENST00000379619,-0.000090
REPIN1_ENST00000425389,-0.000089
RBPJ_ENST00000511401,-0.000089


In [47]:
for idx, t in enumerate(BLSMOTE_features.index):
    if "ZNF385A" in str(t):
        print(idx)

137
1821
1874
2380
2390
2533
2546
5230
7316
9699
9871
11186
11705
12775
12982


In [48]:
for idx, t in enumerate(BLSMOTE_features.index):
    if "CEBPA" in str(t):
        print(idx)

12966


In [49]:
for idx, t in enumerate(BLSMOTE_features.index):
    if "CEBPB" in str(t):
        print(idx)

12989


## Microglia Results

In [50]:
unbalanced_features = node_light_up("unbalanced",transcript_ID,expression,"Microglia")
unbalanced_features.head(n=20) 

/state/partition1/job-21241543/ipykernel_2980960/198994739.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step    
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 873us/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 899us/step


,Microglia
HMGA2_ENST00000403681,-0.005290
TCF7L1_ENST00000282111,-0.003368
ZNF302_ENST00000507959,-0.003097
POU5F1B_ENST00000465342,-0.002919
NFIL3_ENST00000297689,-0.002907
LIN28A_ENST00000326279,-0.002731
ZNF121_ENST00000320451,-0.002726
SOX4_ENST00000244745,-0.002687
SOX11_ENST00000322002,-0.002629
CEBPD_ENST00000408965,-0.002562


In [51]:
for idx, t in enumerate(unbalanced_features.index):
    if "ZNF385A" in str(t):
        print(idx)

307
1152
1199
2072
5449
5788
7677
9185
9787
10857
10912
11504
11557
12067
12196


In [52]:
for idx, t in enumerate(unbalanced_features.index):
    if "CEBPA" in str(t):
        print(idx)

272


In [53]:
for idx, t in enumerate(unbalanced_features.index):
    if "CEBPB" in str(t):
        print(idx)

11495


In [54]:
RO_features = node_light_up("RO",transcript_ID,expression,"Microglia")
RO_features.head(n=20) 

/state/partition1/job-21241543/ipykernel_2980960/198994739.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step    
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 840us/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 856us/step


,Microglia
MYC_ENST00000652288,-0.000112
TCF7L1_ENST00000282111,-0.000083
NME2_ENST00000503064,-0.000081
STAT4_ENST00000392320,-0.000076
YBX1_ENST00000321358,-0.000074
KMT2A_ENST00000686588,-0.000070
EPAS1_ENST00000263734,-0.000066
JUN_ENST00000678696,-0.000066
ZNF428_ENST00000300811,-0.000061
FOXN3_ENST00000615335,-0.000061


In [55]:
for idx, t in enumerate(RO_features.index):
    if "ZNF385A" in str(t):
        print(idx)

1645
5158
5229
5267
5935
6261
7148
9046
10256
11245
12024
12260
12324
12347
12610


In [56]:
for idx, t in enumerate(RO_features.index):
    if "CEBPA" in str(t):
        print(idx)

13063


In [57]:
for idx, t in enumerate(RO_features.index):
    if "CEBPB" in str(t):
        print(idx)

12994


In [58]:
SMOTE_features = node_light_up("SMOTE",transcript_ID,expression,"Microglia")
SMOTE_features.head(n=20) 

/state/partition1/job-21241543/ipykernel_2980960/198994739.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step    
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 873us/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 882us/step


,Microglia
SOX4_ENST00000244745,-0.000036
SON_ENST00000475072,-0.000034
SOX2_ENST00000325404,-0.000033
SOX11_ENST00000322002,-0.000033
HMGA2_ENST00000403681,-0.000032
CAMTA1_ENST00000490738,-0.000022
KLF13_ENST00000621843,-0.000019
ARID3A_ENST00000263620,-0.000018
HMGA1_ENST00000347617,-0.000017
ATF4_ENST00000680446,-0.000017


In [59]:
for idx, t in enumerate(SMOTE_features.index):
    if "ZNF385A" in str(t):
        print(idx)

429
539
1465
1604
2674
4071
5287
6004
7977
8576
8760
11157
11327
11346
11993


In [60]:
for idx, t in enumerate(SMOTE_features.index):
    if "CEBPA" in str(t):
        print(idx)

556


In [61]:
for idx, t in enumerate(SMOTE_features.index):
    if "CEBPB" in str(t):
        print(idx)

12984


In [62]:
BLSMOTE_features = node_light_up("BLSMOTE",transcript_ID,expression,"Microglia")
BLSMOTE_features.head(n=20) 

/state/partition1/job-21241543/ipykernel_2980960/198994739.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step    
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 866us/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 861us/step


,Microglia
SOX2_ENST00000325404,-0.000028
DRAP1_ENST00000532933,-0.000016
HMGA1_ENST00000347617,-0.000013
SOX4_ENST00000244745,-0.000013
HIF1A_ENST00000557446,-0.000012
PA2G4_ENST00000552766,-0.000012
MXD4_ENST00000337190,-0.000012
TCF7_ENST00000395023,-0.000011
CAMTA1_ENST00000490738,-0.000011
TFAP2C_ENST00000201031,-0.000010


In [63]:
for idx, t in enumerate(BLSMOTE_features.index):
    if "ZNF385A" in str(t):
        print(idx)

1145
1185
2318
2608
6370
6791
7907
9640
10730
11646
11847
11941
12248
12470
13126


In [64]:
for idx, t in enumerate(BLSMOTE_features.index):
    if "CEBPA" in str(t):
        print(idx)

13127


In [65]:
for idx, t in enumerate(BLSMOTE_features.index):
    if "CEBPB" in str(t):
        print(idx)

13139
